# **Section 1 - Dataset Overview**

In [ ]:
import pandas as pd

df = pd.read_excel("/content/E Commerce Dataset.xlsx")

df.shape

(5630, 20)

In [ ]:
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

In [ ]:
df.isnull().sum()

,0
CustomerID,0
Churn,0
Tenure,264
PreferredLoginDevice,0
CityTier,0
WarehouseToHome,251
PreferredPaymentMode,0
Gender,0
HourSpendOnApp,255
NumberOfDeviceRegistered,0


# **Section 2 - Data Cleaning**

**Step 1 — Fill Missing Numeric Values**

In [ ]:
numeric_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder"
]

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

/tmp/ipykernel_3412/869130493.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [ ]:
df.isnull().sum()

,0
CustomerID,0
Churn,0
Tenure,0
PreferredLoginDevice,0
CityTier,0
WarehouseToHome,0
PreferredPaymentMode,0
Gender,0
HourSpendOnApp,0
NumberOfDeviceRegistered,0


**Step 3 — Check Duplicate Customers**

In [ ]:
df.duplicated().sum()

np.int64(0)

# **Section 3 - Feature Engineering**

**STEP 1 — Create ChurnFlag**

In [ ]:
df["ChurnFlag"] = df["Churn"]

**STEP 2 — Create TenureGroup**

In [ ]:
df["TenureGroup"] = pd.cut(
    df["Tenure"],
    bins=[-1,6,12,24,100],
    labels=["0-6 Months","7-12 Months","13-24 Months","24+ Months"]
)

**STEP 3 — Create EngagementScore**

In [ ]:
df["EngagementScore"] = (
    df["HourSpendOnApp"] +
    df["OrderCount"] +
    df["CouponUsed"]
) / 3

**STEP 4 — Create RevenueProxy**

In [ ]:
df["RevenueProxy"] = df["OrderCount"] * df["CashbackAmount"]

**STEP 5 — Create RiskSegment**

In [ ]:
df["RiskSegment"] = "Low"

df.loc[
    (df["SatisfactionScore"] <= 2) &
    (df["Complain"] == 1) &
    (df["Tenure"] < 6),
    "RiskSegment"
] = "High"

df.loc[
    (df["SatisfactionScore"] <= 3) &
    (df["Tenure"] < 12),
    "RiskSegment"
] = "Medium"

**STEP 6 — Verify New Columns**

In [ ]:
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,...,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,ChurnFlag,TenureGroup,EngagementScore,RevenueProxy,RiskSegment
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,...,11.0,1.0,1.0,5.0,159.93,1,0-6 Months,1.666667,159.93,Medium
1,50002,1,9.0,Phone,1,8.0,UPI,Male,3.0,4,...,15.0,0.0,1.0,0.0,120.90,1,7-12 Months,1.333333,120.90,Medium
2,50003,1,9.0,Phone,1,30.0,Debit Card,Male,2.0,4,...,14.0,0.0,1.0,3.0,120.28,1,7-12 Months,1.000000,120.28,Medium
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,...,23.0,0.0,1.0,3.0,134.07,1,0-6 Months,1.000000,134.07,Low
4,50005,1,0.0,Phone,1,12.0,CC,Male,3.0,3,...,11.0,1.0,1.0,3.0,129.60,1,0-6 Months,1.666667,129.60,Low


In [ ]:
df["TenureGroup"].value_counts()

,count
TenureGroup,
0-6 Months,2150
7-12 Months,1584
13-24 Months,1467
24+ Months,429


In [ ]:
df.shape

(5630, 25)

**STEP 7 — Export to SQL**

In [ ]:
df.to_csv("ecommerce_churn_clean.csv", index=False)